In [2]:
!mkdir -p data && cd data && aria2c -x16 -s16 -k1M -o Pfam-A.fasta.gz https://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.fasta.gz && gunzip -k Pfam-A.fasta.gz


08/16 22:00:23 [NOTICE] Downloading 1 item(s)
 *** Download Progress Summary as of Sun Aug 16 22:01:34 2026 ***              2s]mm
[#5ff126 5.6GiB/5.8GiB(96%) CN:16 DL:0B]
FILE: /home1/prashantp/data/Pfam-A.fasta.gz
-------------------------------------------------------------------------------

[#5ff126 5.8GiB/5.8GiB(99%) CN:3 DL:60MiB]mm
08/16 22:01:38 [NOTICE] Download complete: /home1/prashantp/data/Pfam-A.fasta.gz

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
5ff126|OK  |    81MiB/s|/home1/prashantp/data/Pfam-A.fasta.gz

Status Legend:
(OK):download completed.


In [1]:
import json
import logging
import math
import os
import time
import itertools as it
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from torch.utils.data import Dataset, DataLoader

In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
logger = logging.getLogger('fela')
if not logger.handlers:
    logger.setLevel(logging.INFO)
    _fmt = logging.Formatter('%(asctime)s %(levelname)s %(message)s')
    _fh = logging.FileHandler('fela.log')
    _fh.setFormatter(_fmt)
    _sh = logging.StreamHandler()
    _sh.setFormatter(_fmt)
    logger.addHandler(_fh)
    logger.addHandler(_sh)

In [32]:
MIN, MAX, OK = 20, 512, set("ACDEFGHIKLMNPQRSTVWYX")
PAD, EOS, UNK = 0, 22, 23

In [ ]:
def seqs(p):
    for k, g in it.groupby(open(p), key=lambda l: l.startswith(">")):
        if not k:
            yield "".join(x.strip().upper() for x in g)

f = open("data/full.fasta", "w")
n = 0
S = dict(total=0, short=0, long=0, bad=0)
for s in seqs("data/Pfam-A.fasta"):
    S["total"] += 1
    if len(s) < MIN: S["short"] += 1
    elif len(s) > MAX: S["long"] += 1
    elif not set(s) <= OK: S["bad"] += 1
    else:
        f.write(s + "\n"); n += 1
logger.info(f"filter stats: {S}, kept: {n}")

In [53]:
AA = "ACDEFGHIKLMNPQRSTVWYX"            # 20 standard AAs + X
stoi = {c: i + 1 for i, c in enumerate(AA)}   # ids 1..21
stoi.update({"<pad>": PAD, "<eos>": EOS, "<unk>": UNK})
itos = {v: k for k, v in stoi.items()}
VOCAB = 32

def encode(seq, max_len=512):
    return [stoi.get(c, UNK) for c in seq[:max_len - 1]] + [EOS]

def decode(ids):
    return "".join(itos.get(i, "?") for i in ids)

json.dump(stoi, open("data/vocab.json", "w"))

In [ ]:
s = "MSDKIIEYDETARRAIEAGVNTLADAV"
logger.info(encode(s))
logger.info(decode(encode(s)))

In [30]:
LUT = np.zeros(256, dtype=np.uint8)

In [ ]:
for c, i in stoi.items():
    if len(c) == 1:
        LUT[ord(c)] = i
        
MAX_LEN = 512
n = sum(1 for l in open("data/full.fasta") if l.strip())
mm = np.memmap("data/full_tokens.dat", dtype=np.uint8, mode="w+", shape=(n, MAX_LEN))
for i, s in enumerate(l.strip() for l in open("data/full.fasta")):
    if not s:
        continue
    ids = LUT[np.frombuffer(s.encode(), dtype=np.uint8)[:MAX_LEN - 1]]
    mm[i, :len(ids)] = ids
    mm[i, len(ids)] = EOS
mm.flush()
logger.info(f"full: {n:,} - {mm.shape}")

In [5]:
class Sin(nn.Module):
    def __init__(self, dim, w=10, train_freq=True):
        super().__init__()
        self.freq = nn.Parameter(w * torch.ones(1, dim)) if train_freq else w * torch.ones(1, dim)
    def forward(self, x):
        return torch.sin(self.freq * x)

class PositionalEmbedding(nn.Module):
    def __init__(self, emb_dim, seq_len):
        super().__init__()
        t = torch.linspace(0, 1, seq_len)[None, :, None]              # 1, L, 1
        bands = (emb_dim - 1) // 2
        t_rescaled = torch.linspace(0, seq_len - 1, seq_len)[None, :, None]
        w = 2 * math.pi * t_rescaled / seq_len                        # 1, L, 1
        f = torch.linspace(1e-4, bands - 1, bands)[None, None]
        z = torch.exp(-1j * f * w)
        z = torch.cat([t, z.real, z.imag], dim=-1)                    # 1, L, emb_dim
        self.register_buffer("z", z)
        self.register_buffer("t", t)
    def forward(self, L):
        return self.z[:, :L], self.t[:, :L]

class ExponentialModulation(nn.Module):
    def __init__(self, d_model, fast_decay_pct=0.3, slow_decay_pct=1.5, target=1e-2, shift=0.0):
        super().__init__()
        self.shift = shift
        max_decay = math.log(target) / fast_decay_pct
        min_decay = math.log(target) / slow_decay_pct
        deltas = torch.linspace(min_decay, max_decay, d_model)[None, None]
        self.register_buffer("deltas", deltas)
    def forward(self, t, x):
        return x * (torch.exp(-t * self.deltas.abs()) + self.shift)

class HyenaFilter(nn.Module):
    def __init__(self, d_model=256, emb_dim=5, order=64, seq_len=514,
                 num_inner_mlps=2, w=10, modulate=True):
        super().__init__()
        self.modulate = modulate
        self.bias = nn.Parameter(torch.randn(d_model))
        act = Sin(dim=order, w=w)
        self.pos_emb = PositionalEmbedding(emb_dim, seq_len)
        self.implicit_filter = nn.Sequential(
            nn.Linear(emb_dim, order), act,
            *[m for _ in range(num_inner_mlps) for m in (nn.Linear(order, order), act)],
            nn.Linear(order, d_model, bias=False),
        )
        self.modulation = ExponentialModulation(d_model)

    def filter(self, L):
        z, t = self.pos_emb(L)
        h = self.implicit_filter(z)                                  # 1, L, d_model
        if self.modulate:
            h = self.modulation(t, h)
        return h

flt = HyenaFilter()
logger.info(flt.filter(128).shape)              # torch.Size([1, 128, 256])
logger.info(flt.filter(514)[:, -1].abs().max()) # envelope at t=1: <= e^-3.07 ~ 0.05

2026-08-17 14:13:36,025 INFO torch.Size([1, 128, 256])
2026-08-17 14:13:36,131 INFO tensor(0.0213, grad_fn=<MaxBackward1>)


In [6]:
class ShortConv(nn.Module):
    def __init__(self, d_model=256, order=2, short_filter_order=3):
        super().__init__()
        total_width = d_model * (order + 1)                 # 768 for order=2
        self.in_proj = nn.Linear(d_model, total_width)      # [B, L, 768]
        self.conv = nn.Conv1d(total_width, total_width, short_filter_order,
                              groups=total_width,           # depthwise
                              padding=short_filter_order - 1)
    def forward(self, u):
        u = self.in_proj(u)                    # [B, L, 768]
        u = u.transpose(1, 2)                  # [B, 768, L]
        u = self.conv(u)[..., :u.shape[-1]]    # left-pad -> causal, keep L
        return u                               # [B, 768, L]

sc = ShortConv()
x = torch.randn(4, 128, 256)
logger.info(sc(x).shape)                            # torch.Size([4, 768, 128])

2026-08-17 14:13:37,398 INFO torch.Size([4, 768, 128])


In [7]:
def fft_conv(u, k, bias=None):
    seqlen = u.shape[-1]
    fft_size = 2 * seqlen
    k_f = torch.fft.rfft(k, n=fft_size) / fft_size
    if len(u.shape) > 3:
        k_f = k_f.unsqueeze(1)                          # broadcast over head dim
    u_f = torch.fft.rfft(u.to(dtype=k.dtype), n=fft_size)
    y = torch.fft.irfft(u_f * k_f, n=fft_size, norm="forward")[..., :seqlen]
    return y + u * bias.unsqueeze(-1) if bias is not None else y

def naive_conv(u, k):
    L = u.shape[-1]
    out = torch.zeros_like(u)
    for n in range(L):
        for i in range(n + 1):
            out[:, :, n] += k[:, i] * u[:, :, n - i]
    return out

L = 128
u = torch.randn(4, 256, L)
t = torch.arange(L, dtype=torch.float32)
k = torch.randn(256, L) * torch.exp(-t).unsqueeze(0)    # decaying filter
bias = torch.randn(256)

err = (fft_conv(u, k, bias) - (naive_conv(u, k) + u * bias.unsqueeze(-1))).abs().max().item()
logger.info(f"max |fft - naive| = {err}")

2026-08-17 14:13:38,061 INFO max |fft - naive| = 2.86102294921875e-06


In [8]:
class HyenaOperator(nn.Module):
    def __init__(self, d_model=256, l_max=514, order=2, filter_order=64,
                 short_filter_order=3, drop_rate=0.0):
        super().__init__()
        self.d_model, self.order, self.l_max = d_model, order, l_max
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        total_width = d_model * (order + 1)
        self.short_filter = nn.Conv1d(total_width, total_width, short_filter_order,
                                      groups=total_width, padding=short_filter_order - 1)
        self.filter_fn = HyenaFilter(d_model=d_model, order=filter_order, seq_len=l_max)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, u):
        l_filter = min(u.size(-2), self.l_max)
        u = rearrange(self.in_proj(u), "b l d -> b d l")       # [B, 768, L]
        uc = self.short_filter(u)[..., :l_filter]
        *x, v = uc.split(self.d_model, dim=1)                  # 768 -> 3 chunks of 256: x0, x1, v

        k = self.filter_fn.filter(l_filter)                    # [1, L, 256]
        k = rearrange(k, "c l (v o) -> c o v l", v=self.d_model, o=self.order - 1)
        bias = rearrange(self.filter_fn.bias, "(v o) -> o v", o=self.order - 1)

        for o, x_i in enumerate(reversed(x[1:])):
            v = self.dropout(v * x_i)
            v = fft_conv(v, k[o], bias[o])                     # [B, 256, L]

        return self.out_proj(rearrange(v * x[0], "b v l -> b l v"))

def naive_longconv(v, k, bias):
    B, C, L = v.shape
    out = torch.zeros_like(v)
    k0 = k[0]
    for n in range(L):
        for i in range(n + 1):
            out[:, :, n] += k0[:, i] * v[:, :, n - i]
    return out + v * bias.unsqueeze(-1)

def hyena_naive(u, op):
    B, L, D = u.shape
    u = rearrange(op.in_proj(u), "b l d -> b d l")
    uc = op.short_filter(u)[..., :L]
    *x, v = uc.split(D, dim=1)                                 # same split as the reference
    k = rearrange(op.filter_fn.filter(L), "c l (v o) -> c o v l", v=D, o=op.order - 1)
    bias = rearrange(op.filter_fn.bias, "(v o) -> o v", o=op.order - 1)
    for o, x_i in enumerate(reversed(x[1:])):
        v = v * x_i
        v = naive_longconv(v, k[o], bias[o])
    return op.out_proj(rearrange(v * x[0], "b v l -> b l v"))

torch.manual_seed(0)
op = HyenaOperator().eval()
u = torch.randn(4, 128, 256)
with torch.no_grad():
    y1, y2 = op(u), hyena_naive(u, op)
logger.info(f"op out: {y1.shape}")
logger.info(f"max |fft-op - naive-op| = {(y1 - y2).abs().max().item()}")

2026-08-17 14:13:39,535 INFO op out: torch.Size([4, 128, 256])
2026-08-17 14:13:39,541 INFO max |fft-op - naive-op| = 3.5762786865234375e-07


In [9]:
class _Block(nn.Module):                       # flash_attn Block, prenorm=True, drop_path=0
    def __init__(self, d_model, d_inner, l_max, drop1_p, drop2_p, eps=1e-5, residual_in_fp32=True):
        super().__init__()
        self.drop1 = nn.Dropout(drop1_p)
        self.norm1 = nn.LayerNorm(d_model, eps=eps)
        self.mixer = HyenaOperator(d_model=d_model, l_max=l_max)
        self.drop2 = nn.Dropout(drop2_p)
        self.norm2 = nn.LayerNorm(d_model, eps=eps)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_inner),
                                 nn.GELU(approximate="tanh"),
                                 nn.Linear(d_inner, d_model))
        self.residual_in_fp32 = residual_in_fp32

    def forward(self, hidden, residual):
        dropped = self.drop1(hidden)
        residual = (dropped + residual) if residual is not None else dropped
        hidden = self.mixer(self.norm1(residual.to(dtype=self.norm1.weight.dtype)))
        if self.residual_in_fp32:
            residual = residual.float()
        dropped = self.drop2(hidden)
        residual = (dropped + residual) if residual is not None else dropped
        hidden = self.mlp(self.norm2(residual.to(dtype=self.norm2.weight.dtype)))
        if self.residual_in_fp32:
            residual = residual.float()
        return hidden, residual

class Fela(nn.Module):
    def __init__(self, d_model=256, n_layer=2, d_inner=1024, vocab_size=32, l_max=514,
                 embed_dropout=0.1, resid_dropout=0.0, eps=1e-5, residual_in_fp32=True):
        super().__init__()
        torch.manual_seed(2222)
        self.embed = nn.Embedding(vocab_size, d_model)             
        self.blocks = nn.ModuleList(
            _Block(d_model, d_inner, l_max,
                   drop1_p=embed_dropout if i == 0 else resid_dropout,
                   drop2_p=resid_dropout, eps=eps,
                   residual_in_fp32=residual_in_fp32)
            for i in range(n_layer)
        )
        self.drop_f = nn.Dropout(resid_dropout)
        self.ln_f = nn.LayerNorm(d_model, eps=eps)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights(n_layer)                               
        self.lm_head.weight = self.embed.weight                     # tie

    def _init_weights(self, n_layer):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
        for name, p in self.named_parameters():
            if name.endswith("out_proj.weight") or name.endswith("mlp.2.weight"):
                nn.init.normal_(p, std=0.02 / math.sqrt(2 * n_layer))

    def forward(self, input_ids):
        hidden = self.embed(input_ids)
        residual = None
        for block in self.blocks:
            hidden, residual = block(hidden, residual)
        dropped = self.drop_f(hidden)
        residual = (dropped + residual) if residual is not None else dropped
        hidden = self.ln_f(residual.to(dtype=self.ln_f.weight.dtype))
        return self.lm_head(hidden)

In [10]:
model = Fela()
n_params = sum(p.numel() for p in model.parameters())
logger.info(f"params: {n_params/1e6:.3f}M")
x = torch.randint(1, 22, (4, 128))
with torch.no_grad():
    logits = model(x)
    logger.info(f"logits: {logits.shape}")
    ce = torch.nn.functional.cross_entropy(logits[:, :-1].reshape(-1, 32), x[:, 1:].reshape(-1))
    logger.info(f"init CE (expect ~ln 32 = 3.466): {ce.item()}")

2026-08-17 14:13:43,312 INFO params: 1.645M
2026-08-17 14:13:43,743 INFO logits: torch.Size([4, 128, 32])
2026-08-17 14:13:43,752 INFO init CE (expect ~ln 32 = 3.466): 3.5579583644866943


In [11]:
def naive_longconv(v, k, bias):                 
    B, C, L = v.shape
    out = torch.zeros_like(v)
    k0 = k[0]
    for n in range(L):
        for i in range(n + 1):
            out[:, :, n] += k0[:, i] * v[:, :, n - i]
    return out + v * bias.unsqueeze(-1)

def fela_naive(model, x):                        # naive long convs
    hidden = model.embed(x)
    residual = None
    for block in model.blocks:
        dropped = block.drop1(hidden)
        residual = (dropped + residual) if residual is not None else dropped
        h = block.norm1(residual.to(dtype=block.norm1.weight.dtype))
        if block.residual_in_fp32:
            residual = residual.float()
        m = block.mixer
        u = rearrange(m.in_proj(h), "b l d -> b d l")
        uc = m.short_filter(u)[..., :h.shape[1]]
        *xp, v = uc.split(m.d_model, dim=1)
        k = rearrange(m.filter_fn.filter(h.shape[1]), "c l (v o) -> c o v l",
                      v=m.d_model, o=m.order - 1)
        bias = rearrange(m.filter_fn.bias, "(v o) -> o v", o=m.order - 1)
        for o, xi in enumerate(reversed(xp[1:])):
            v = m.dropout(v * xi)
            v = naive_longconv(v, k[o], bias[o])
        hidden = m.out_proj(rearrange(v * xp[0], "b v l -> b l v"))
        dropped = block.drop2(hidden)
        residual = (dropped + residual) if residual is not None else dropped
        hidden = block.mlp(block.norm2(residual.to(dtype=block.norm2.weight.dtype)))
        if block.residual_in_fp32:
            residual = residual.float()
    dropped = model.drop_f(hidden)
    residual = (dropped + residual) if residual is not None else dropped
    hidden = model.ln_f(residual.to(dtype=model.ln_f.weight.dtype))
    return model.lm_head(hidden)

model.eval()
for L in (128, 256, 512):
    x = torch.randint(1, 22, (4, L))
    with torch.no_grad():
        y1, y2 = model(x), fela_naive(model, x)
    logger.info(f"L={L:4d}  max |fft-LM - naive-LM| = {(y1 - y2).abs().max().item():.3e}")

2026-08-17 14:13:54,320 INFO L= 128  max |fft-LM - naive-LM| = 7.153e-07
2026-08-17 14:13:56,025 INFO L= 256  max |fft-LM - naive-LM| = 7.749e-07
2026-08-17 14:14:04,710 INFO L= 512  max |fft-LM - naive-LM| = 1.073e-06


In [18]:
SEQ_LEN = 512
BATCH_SIZE = 128

def load_rows(path):
    if path.endswith('.dat'):
        n = os.path.getsize(path) // SEQ_LEN                      # raw uint8 memmap
        return np.memmap(path, dtype=np.uint8, mode='r', shape=(n, SEQ_LEN))
    return np.load(path, mmap_mode='r')                            # .npy

def pack(rows, out_path):
    if os.path.exists(out_path):
        return np.load(out_path, mmap_mode='r')
    chunks = []
    for s in range(0, rows.shape[0], 100000):                     # chunked
        r = np.asarray(rows[s:s + 100000])
        chunks.append(np.concatenate([row[row != 0] for row in r]))
    packed = np.concatenate(chunks).astype(np.uint8)
    np.save(out_path, packed)
    return packed

In [ ]:
full = pack(load_rows('data/full_tokens.dat'), 'data/full_packed.npy')
logger.info(f"train tokens: {full.size}")

In [34]:
class LMDataset(Dataset):                                        
    def __init__(self, tokens, seq_len):
        self.seq_len = seq_len
        ntokens = ((len(tokens) - 1) // seq_len) * seq_len + 1     # drop_last
        self.ntokens = ntokens
        self.tokens = tokens
        self.total = math.ceil((self.ntokens - 1) / seq_len)
    def __len__(self):
        return self.total
    def __getitem__(self, idx):
        s = idx * self.seq_len
        l = min(self.seq_len, self.ntokens - 1 - s)
        data = torch.as_tensor(self.tokens[s:s + l + 1].astype(np.int64))
        return data[:-1], data[1:].clone()

In [ ]:
train_ds = LMDataset(full, SEQ_LEN)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=4, drop_last=True)
logger.info(f"train steps/epoch: {len(train_dl)}")

In [ ]:
torch.manual_seed(2222); np.random.seed(2222)
torch.set_float32_matmul_precision('high')
model = Fela().to(device)
logger.info(f"device: {device} | params: {sum(p.numel() for p in model.parameters())/1e6:.3f}M")

BATCH_SIZE = 128
GLOBAL_BATCH = 256
ACCUM = GLOBAL_BATCH // BATCH_SIZE                     # 2
MAX_STEPS = 40000
WARMUP = int(0.01 * MAX_STEPS)                         # 400  (config: warmup_t = 1% of t_initial)
LR, LR_MIN, WARMUP_LR_INIT = 6e-4, 6e-5, 1e-6          # lr_min = 0.1*lr (config)
CLIP = 1.0
USE_BF16 = True                                        # Blackwell bf16 tensor cores

def lr_at(t):                                          # timm cosine_warmup
    if t < WARMUP:
        return WARMUP_LR_INIT + (LR - WARMUP_LR_INIT) * t / WARMUP
    p = (t - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return LR_MIN + 0.5 * (LR - LR_MIN) * (1 + math.cos(math.pi * p))

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=USE_BF16)

def run_batch(x, y, train):
    x, y = x.to(device), y.to(device)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, 32), y.reshape(-1)) / ACCUM
    if train:
        scaler.scale(loss).backward()
    return loss.item() * ACCUM

model.train()
it = iter(train_dl)
t0 = time.time()
for step in range(1, MAX_STEPS + 1):
    opt.zero_grad(set_to_none=True)
    acc, cnt = 0.0, 0
    for _ in range(ACCUM):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_dl); x, y = next(it)
        acc += run_batch(x, y, True); cnt += y.numel()
    opt.param_groups[0]['lr'] = lr_at(step)
    scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), CLIP)
    scaler.step(opt); scaler.update()

    if step % 500 == 0:
        logger.info(f"step {step:6d} | lr {lr_at(step):.2e} | loss {acc/ACCUM:.4f} | ppl {math.exp(acc/ACCUM):.3f} | {cnt/(time.time()-t0)/1e3:.0f}k tok/s")
        t0 = time.time()
    if step % 2000 == 0:
        torch.save(model.state_dict(), 'fela_checkpoint.pt')
        logger.info(f"step {step}: saved fela_checkpoint.pt")
    if step % 10000 == 0:
        torch.save(model.state_dict(), f'fela_{step}.pt')
        logger.info(f"step {step}: saved fela_{step}.pt")